In [0]:
import json 
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS load_json")

spark.sql("USE CATALOG load_json")

spark.sql("CREATE SCHEMA IF NOT EXISTS load_json.bronze")

spark.sql("CREATE VOLUME IF NOT EXISTS load_json.bronze.raw")



In [0]:
file_path = "/Volumes/load_json/bronze/raw/nested_json_data_cleaning_practice.json"

display(dbutils.fs.ls("/Volumes/load_json/bronze/raw/"))

In [0]:
raw_df = spark.read.format("json").option("multiline", "true").load(file_path)
display(raw_df)
raw_df.printSchema()

In [0]:
bronze_df = (
    raw_df
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

display(bronze_df)

In [0]:
bronze_table = "load_json.bronze.raw_json_documents"

(
    bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

In [0]:
bronze_check = spark.table(
    "load_json.bronze.raw_json_documents"
)

display(bronze_check)

bronze_check.printSchema()

print("Bronze document count:", bronze_check.count())